# Alpha Vantage News Sentiment Analysis
**Professional-Grade Financial News Sentiment using Alpha Vantage API**

This notebook uses Alpha Vantage's News Sentiment API for sentiment analysis.

## Features:
- Alpha Vantage News Sentiment API
- Professional sentiment scoring (-1 to +1)
- Ticker-specific relevance scores
- Trading recommendations with confidence

**API Key Required:** Get free key at https://www.alphavantage.co/support/#api-key

In [1]:
# pip install requests pandas
import requests
import pandas as pd
from typing import Dict, Tuple
from datetime import datetime, timedelta
import time
print('Imports complete')

In [2]:
class AlphaVantageNewsAPI:
    BASE_URL = 'https://www.alphavantage.co/query'
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.request_count = 0
    def get_news_sentiment(self, tickers=None, limit=50):
        params = {'function': 'NEWS_SENTIMENT', 'apikey': self.api_key, 'limit': limit}
        if tickers:
            params['tickers'] = ','.join(t.upper() for t in tickers)
        try:
            response = requests.get(self.BASE_URL, params=params, timeout=30)
            data = response.json()
            if 'feed' in data:
                print(f'Fetched {len(data["feed"])} articles')
                return data
            return {'feed': []}
        except Exception as e:
            print(f'Error: {e}')
            return {'feed': []}

class NewsDataProcessor:
    @staticmethod
    def parse_news_items(api_response: Dict, target_ticker=None):
        if not api_response.get('feed'):
            return pd.DataFrame()
        processed_items = []
        for item in api_response['feed']:
            overall_score = float(item.get('overall_sentiment_score', 0))
            ticker_sentiment = None
            if target_ticker and 'ticker_sentiment' in item:
                for ts in item['ticker_sentiment']:
                    if ts.get('ticker', '').upper() == target_ticker.upper():
                        ticker_sentiment = float(ts.get('ticker_sentiment_score', 0))
                        break
            final_sentiment = ticker_sentiment if ticker_sentiment is not None else overall_score
            processed_items.append({
                'Title': item.get('title', 'NA'),
                'Source': item.get('source', 'Unknown'),
                'Sentiment': final_sentiment
            })
        return pd.DataFrame(processed_items)

class TradingRecommendationEngine:
    THRESHOLDS = {'strong_buy': 0.25, 'buy': 0.15, 'sell': -0.15, 'strong_sell': -0.25}
    def calculate_metrics(self, df):
        if df.empty:
            return {}
        scores = df['Sentiment']
        return {
            'avg_sentiment': scores.mean(),
            'positive_ratio': sum(scores >= 0.35) / len(scores),
            'total_articles': len(scores)
        }
    def generate_recommendation(self, metrics):
        avg = metrics['avg_sentiment']
        if avg >= self.THRESHOLDS['strong_buy']:
            return 'STRONG BUY', f'Strong positive sentiment {avg:.3f}'
        elif avg >= self.THRESHOLDS['buy']:
            return 'BUY', f'Positive sentiment {avg:.3f}'
        elif avg <= self.THRESHOLDS['strong_sell']:
            return 'STRONG SELL', f'Strong negative sentiment {avg:.3f}'
        elif avg <= self.THRESHOLDS['sell']:
            return 'SELL', f'Negative sentiment {avg:.3f}'
        else:
            return 'NEUTRAL', f'Mixed sentiment {avg:.3f}'

print('Classes ready')

In [3]:
# Example usage
API_KEY = 'YOUR_API_KEY_HERE'  # Replace with your key

api_client = AlphaVantageNewsAPI(api_key=API_KEY)
processor = NewsDataProcessor()
rec_engine = TradingRecommendationEngine()

ticker = 'AAPL'
api_response = api_client.get_news_sentiment(tickers=[ticker], limit=30)
df = processor.parse_news_items(api_response, target_ticker=ticker)

if not df.empty:
    metrics = rec_engine.calculate_metrics(df)
    recommendation, rationale = rec_engine.generate_recommendation(metrics)
    print(f'\nAnalysis for {ticker}:')
    print(f'Articles: {metrics["total_articles"]}')
    print(f'Avg Sentiment: {metrics["avg_sentiment"]:.4f}')
    print(f'Recommendation: {recommendation}')
    print(f'Rationale: {rationale}')
else:
    print('No data available')